# 01 — `midas-uq` Quickstart: four UQ diagnostics on one synthetic grain

`midas-uq` answers, for a refined HEDM grain and *without any ground
truth*:

1. **`half_half`** — how reproducible is the refined state under random
   K-splits of the measured spots? (empirical posterior spread)
2. **`jackknife`** — which individual spots drive the fit, and which look
   corrupted? (leave-one-out influence)
3. **`laplace_covariance`** — what is the local Gaussian posterior from
   the inverse Hessian? (analytic baseline)
4. **`rfree_gap`** — is the refinement overfitting in the low-spots-per-DOF
   regime? (train vs holdout loss)

We **plant** a known FCC grain, forward-simulate its spots with
`midas-diffract`, treat those as the "measurement", perturb the seed,
and run each diagnostic. Everything is CPU + synthetic and runs in well
under a minute.


In [1]:
# midas-diffract / numpy share an OpenMP runtime with torch; allow the
# duplicate load so the import doesn't abort on macOS.
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import math
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)
DEG2RAD = math.pi / 180.0
DEVICE = "cpu"          # this package is CPU-only here
torch.set_default_dtype(torch.float64)
print("torch", torch.__version__, "| device:", DEVICE)


torch 2.11.0 | device: cpu


## Step 1 — Build the forward model and plant a grain

We use the same paper-I FF geometry the package tests use: a 2048² panel
at 1 m, 0.25° ω steps, FCC Au (a = 4.08 Å) at 71.7 keV. The reflection
list comes from `midas-hkls` (no C `GetHKLList` dependency).


In [2]:
from midas_diffract import HEDMForwardModel, HEDMGeometry, hkls_for_forward_model
from midas_hkls import SpaceGroup, Lattice

geom = HEDMGeometry(
    Lsd=1_000_000.0, y_BC=1024.0, z_BC=1024.0, px=200.0,
    omega_start=0.0, omega_step=0.25, n_frames=1440,
    n_pixels_y=2048, n_pixels_z=2048,
    min_eta=6.0, wavelength=0.172979,
)
sg = SpaceGroup.from_number(225)              # FCC
lat = Lattice.for_system("cubic", a=4.08)     # Au
hkls_cart, thetas, hkls_int = hkls_for_forward_model(
    sg, lat, wavelength_A=geom.wavelength, two_theta_max_deg=15.0,
)
model = HEDMForwardModel(hkls=hkls_cart, thetas=thetas,
                         geometry=geom, hkls_int=hkls_int)

gt_euler = torch.tensor([45.0, 30.0, 60.0]) * DEG2RAD
gt_latc  = torch.tensor([4.08, 4.08, 4.08, 90.0, 90.0, 90.0])
gt_pos   = torch.zeros(3)
print("planted grain:", gt_euler.tolist(), "rad")


planted grain: [0.7853981633974483, 0.5235987755982988, 1.0471975511965976] rad


## Step 2 — Synthesise the "measured" spots

Forward-simulate the planted grain and keep only the valid spots in
**angular** space `(2θ, η, ω)` in radians — this is exactly the
observation format `half_half`/`jackknife`/`laplace`/`rfree` expect for
FF and pf modes.


In [3]:
spots = model(gt_euler.unsqueeze(0), gt_pos.unsqueeze(0), lattice_params=gt_latc)
ang, valid = HEDMForwardModel.predict_spot_coords(spots, space="angular")
obs = ang.squeeze()[valid.squeeze() > 0.5].detach().clone()
print(f"valid spots (observations): {obs.shape[0]}")
assert obs.shape[0] >= 12, "need enough spots for K-splitting"


valid spots (observations): 158


## Step 3 — A perturbed seed

A real workflow seeds `midas-uq` from an indexer/Grains.csv row, not the
truth. We mimic that by perturbing the orientation by ~0.5° and the
lattice by ~0.001 Å.


In [4]:
import midas_uq as muq

init = muq.GrainState(
    euler_rad=gt_euler + 0.5 * DEG2RAD * torch.randn(3),
    latc=gt_latc + 1e-3 * torch.randn(6),
    pos=gt_pos,
)
print("seed euler perturbation (deg):",
      np.round(((init.euler_rad - gt_euler) / DEG2RAD).tolist(), 3))


seed euler perturbation (deg): [ 0.77  -0.147 -1.089]


## Diagnostic 1 — `half_half` (K-split reproducibility)

Each of `n_splits` random partitions refines two independent halves of
the spot set; the disagreement between the two refined states is the
empirical UQ. `mode="ff"` (and `"pf"`) use the spot-based path; `"nf"`
would use frames.


In [5]:
hh = muq.half_half(model, init, obs, mode="ff", n_splits=5, phase_steps=(8, 8, 5))
print("per-split misorientation disagreement (deg):",
      np.round(hh.misori_deg, 4))
print(f"misori   median={hh.misori_median_deg:.4f}  p90={hh.misori_p90_deg:.4f} deg")
print(f"lattice  median={hh.lattice_median_A:.2e}  p90={hh.lattice_p90_A:.2e} A")
print("\nInterpretation: small, tight spread => the refinement is "
      "reproducible under resampling (well-determined grain).")


per-split misorientation disagreement (deg): [0.0023 0.0034 0.0011 0.0017 0.0014]
misori   median=0.0017  p90=0.0030 deg
lattice  median=1.27e-04  p90=3.17e-04 A

Interpretation: small, tight spread => the refinement is reproducible under resampling (well-determined grain).


## Diagnostic 2 — `jackknife` (per-spot influence)

Leave-one-out: refit dropping each spot in turn, measure how far the
refined orientation moves. High-influence spots are leverage points or
corruption candidates — the drill-down on grains `half_half` flags.


In [6]:
jk = muq.jackknife(model, init, obs, mode="ff")
top = jk.top_k(5, by="misori")
print("5 most influential spots (index, influence in deg):")
for i in top:
    print(f"  spot {int(i):3d}:  {jk.influence_misori_deg[int(i)]:.4e} deg")
print(f"\nmean influence = {jk.influence_misori_deg.mean():.4e} deg  "
      f"(uniform low influence => no single spot dominates the fit)")


5 most influential spots (index, influence in deg):
  spot  52:  7.3306e-04 deg
  spot 153:  7.0291e-04 deg
  spot 110:  6.9963e-04 deg
  spot 123:  6.9849e-04 deg
  spot 148:  6.9645e-04 deg

mean influence = 6.8154e-04 deg  (uniform low influence => no single spot dominates the fit)


## Diagnostic 3 — `laplace_covariance` (Hessian baseline)

The Laplace approximation builds the local Gaussian posterior on
`(euler | latc)` from the inverse Hessian of the NLL at the converged
state. `sigma_vec` is the per-coordinate measurement noise
(σ_2θ, σ_η, σ_ω) in radians. We pass `refine_first=True` so it converges
the perturbed seed before forming the Hessian.


In [7]:
sigma_vec = torch.tensor([0.5, 0.5, 0.5]) * DEG2RAD * 0.1   # ~0.05 deg noise
lp = muq.laplace_covariance(model, init, obs, sigma_vec,
                            refine_first=True, n_mc_samples=1000)
print(f"misori  p95 = {lp.misori_p95_deg:.4f} deg")
print(f"lattice p95 = {lp.lattice_p95_A:.2e} A")
print(f"Hessian condition number = {lp.condition_number:.3e}")
print("\nInterpretation: a Laplace p95 that disagrees with the "
      "half_half spread is itself a flag for a non-Gaussian / "
      "multi-basin posterior.")


misori  p95 = 0.0201 deg
lattice p95 = 2.90e-03 A
Hessian condition number = 4.397e+04

Interpretation: a Laplace p95 that disagrees with the half_half spread is itself a flag for a non-Gaussian / multi-basin posterior.


## Diagnostic 4 — `rfree_gap` (overfitting check)

Split the spots into a train and a holdout set, refine on train only, and
track both losses across the three-phase L-BFGS. A holdout loss that
sits far above the train loss (large `gap_final`) means the refinement is
explaining noise — the low-spots-per-DOF overfitting regime.


In [8]:
rf = muq.rfree_gap(model, init, obs, train_fraction=0.5)
print(f"n_train={rf.n_train}  n_hold={rf.n_hold}")
print(f"final train loss   = {rf.train_losses[-1]:.4e}")
print(f"final holdout loss = {rf.holdout_losses[-1]:.4e}")
print(f"R_free-style gap   = {rf.gap_final:.3f}  "
      "(near 0 => generalises; large => overfit)")


n_train=79  n_hold=79
final train loss   = 1.6913e-09
final holdout loss = 1.4476e-09
R_free-style gap   = -0.144  (near 0 => generalises; large => overfit)


## Mode dispatch: `ff` / `pf` / `nf`

`half_half` and `jackknife` are mode-aware. `"ff"` and `"pf"` share the
spot-based implementation (observations are `(N, 3)` angular coords);
`"nf"` switches to the frame-based path (observations are an `(F, H, W)`
image stack). Unknown modes raise.


In [9]:
# ff and pf accept the identical spot observation tensor:
hh_pf = muq.half_half(model, init, obs, mode="pf", n_splits=3, phase_steps=(6, 6, 4))
print("pf-mode half_half median misori (deg):",
      round(hh_pf.misori_median_deg, 4))

try:
    muq.half_half(model, init, obs, mode="garbage")
except ValueError as e:
    print("unknown mode correctly rejected:", str(e)[:60], "...")


pf-mode half_half median misori (deg): 0.0023
unknown mode correctly rejected: Unknown mode: 'garbage'. Use one of {'pf', 'nf', 'ff'}. ...


## Summary

| Diagnostic | What it surfaces | Cost |
|---|---|---|
| `rfree_gap` | overfitting at low n_obs/DOF | 1× refine |
| `half_half` | noise + model misspec + local-minimum basin | ~10× refine |
| `jackknife` | per-spot leverage / corruption | N_obs × refine |
| `laplace_covariance` | local Gaussian baseline | 1× Hessian |

Recommended population workflow: `half_half` over all grains, then
`jackknife` on the flagged ones, with `laplace_covariance` as a
complementary Gaussian baseline. See `dev/paper/` in the repo for the
Ti-7Al / Park22 population studies.
